# Training Record Validation

This notebook validates employee training records for the complete synthetic workforce.

The main rules are:

- Training-record IDs are complete and unique.
- Employees and training programs must exist.
- Training cannot begin before hire or after employment ends.
- Completion dates must follow start dates.
- Completed and failed programs require completion dates.
- Incomplete programs must not have completion dates.
- Completed programs require sufficient training hours.
- Scores must agree with completion status.
- Every employee receives exactly one onboarding record.
- Managers receive leadership training.
- Safety-required employee groups receive safety training.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

employees = pd.read_csv(
    RAW_DATA_DIR / "employees.csv",
    parse_dates=[
        "hire_date",
        "termination_date",
    ],
)

training_programs = pd.read_csv(
    RAW_DATA_DIR / "training_programs.csv"
)

training_records = pd.read_csv(
    RAW_DATA_DIR / "training_records.csv",
    parse_dates=[
        "start_date",
        "completion_date",
    ],
)

print("Employees:", employees.shape)
print(
    "Training programs:",
    training_programs.shape,
)
print(
    "Training records:",
    training_records.shape,
)

Employees: (10000, 15)
Training programs: (10, 5)
Training records: (26433, 8)


## 1. Initial inspection

In [2]:
training_records.head(10)

,training_record_id,employee_id,program_id,start_date,completion_date,completion_status,training_hours,score
0,900001,100001,1,2022-10-27,2022-11-06,Completed,8.1,84.3
1,900002,100001,4,2023-07-27,2023-08-10,Completed,20.1,86.9
2,900003,100001,5,2023-07-07,2023-07-19,Completed,17.0,79.9
3,900004,100001,6,2025-09-21,2025-09-29,Completed,26.5,88.0
4,900005,100002,1,2022-04-04,2022-04-11,Completed,8.7,90.5
5,900006,100002,2,2022-08-18,2022-08-26,Completed,6.1,73.9
6,900007,100002,4,2023-03-29,2023-04-12,Completed,22.7,82.6
7,900008,100002,5,2022-08-12,2022-08-19,Completed,17.9,90.0
8,900009,100002,7,2025-07-04,2025-07-15,Completed,36.2,79.6
9,900010,100003,1,2021-07-09,2021-07-21,Completed,8.1,92.0


In [3]:
training_records.info()

<class 'pandas.DataFrame'>
RangeIndex: 26433 entries, 0 to 26432
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   training_record_id  26433 non-null  int64         
 1   employee_id         26433 non-null  int64         
 2   program_id          26433 non-null  int64         
 3   start_date          26433 non-null  datetime64[us]
 4   completion_date     24563 non-null  datetime64[us]
 5   completion_status   26433 non-null  str           
 6   training_hours      26433 non-null  float64       
 7   score               24563 non-null  float64       
dtypes: datetime64[us](2), float64(2), int64(3), str(1)
memory usage: 1.6 MB


## 2. Connect employees and programs

In [4]:
employee_details = (
    employees[
        [
            "employee_id",
            "hire_date",
            "termination_date",
            "department_id",
            "employment_type",
            "organizational_level",
        ]
    ]
    .copy()
)

employee_details[
    "employment_end_date"
] = (
    employee_details[
        "termination_date"
    ]
    .fillna(
        pd.Timestamp("2026-06-30")
    )
)

training_details = (
    training_records
    .merge(
        employee_details,
        on="employee_id",
        how="left",
    )
    .merge(
        training_programs,
        on="program_id",
        how="left",
    )
)

training_details.head()

,training_record_id,employee_id,program_id,start_date,completion_date,completion_status,training_hours,score,hire_date,termination_date,department_id,employment_type,organizational_level,employment_end_date,program_name,program_category,required_hours,mandatory
0,900001,100001,1,2022-10-27,2022-11-06,Completed,8.1,84.3,2022-10-17,NaT,1,Salaried,Department Head,2026-06-30,New Employee Orientation,Onboarding,8,True
1,900002,100001,4,2023-07-27,2023-08-10,Completed,20.1,86.9,2022-10-17,NaT,1,Salaried,Department Head,2026-06-30,Python for Data Analysis,Technical,20,False
2,900003,100001,5,2023-07-07,2023-07-19,Completed,17.0,79.9,2022-10-17,NaT,1,Salaried,Department Head,2026-06-30,SQL Fundamentals,Technical,16,False
3,900004,100001,6,2025-09-21,2025-09-29,Completed,26.5,88.0,2022-10-17,NaT,1,Salaried,Department Head,2026-06-30,First-Time Manager Program,Leadership,24,False
4,900005,100002,1,2022-04-04,2022-04-11,Completed,8.7,90.5,2022-04-03,NaT,1,Salaried,Senior Manager,2026-06-30,New Employee Orientation,Onboarding,8,True


## 3. Primary-key and foreign-key checks

In [5]:
key_checks = pd.Series(
    {
        "training-record IDs are complete": (
            training_records[
                "training_record_id"
            ].notna().all()
        ),
        "training-record IDs are unique": (
            training_records[
                "training_record_id"
            ].is_unique
        ),
        "employee IDs are valid": (
            set(
                training_records[
                    "employee_id"
                ]
            )
            .issubset(
                set(
                    employees[
                        "employee_id"
                    ]
                )
            )
        ),
        "program IDs are valid": (
            set(
                training_records[
                    "program_id"
                ]
            )
            .issubset(
                set(
                    training_programs[
                        "program_id"
                    ]
                )
            )
        ),
        "employee-program combinations are unique": (
            not training_records
            .duplicated(
                subset=[
                    "employee_id",
                    "program_id",
                ]
            )
            .any()
        ),
        "all employees have training records": (
            training_records[
                "employee_id"
            ].nunique()
            == len(employees)
        ),
    },
    name="passed",
)

key_checks

training-record IDs are complete            True
training-record IDs are unique              True
employee IDs are valid                      True
program IDs are valid                       True
employee-program combinations are unique    True
all employees have training records         True
Name: passed, dtype: bool

## 4. Training-date checks

In [6]:
completed_or_failed_mask = (
    training_details[
        "completion_status"
    ]
    .isin(
        [
            "Completed",
            "Failed",
        ]
    )
)

incomplete_mask = (
    training_details[
        "completion_status"
    ]
    == "Incomplete"
)

dated_training = training_details[
    training_details[
        "completion_date"
    ].notna()
]

date_checks = pd.Series(
    {
        "training does not start before hire": (
            training_details[
                "start_date"
            ]
            .ge(
                training_details[
                    "hire_date"
                ]
            )
            .all()
        ),
        "training does not start after employment": (
            training_details[
                "start_date"
            ]
            .le(
                training_details[
                    "employment_end_date"
                ]
            )
            .all()
        ),
        "completed and failed records have completion dates": (
            training_details.loc[
                completed_or_failed_mask,
                "completion_date",
            ].notna().all()
        ),
        "incomplete records have no completion dates": (
            training_details.loc[
                incomplete_mask,
                "completion_date",
            ].isna().all()
        ),
        "completion does not occur before start": (
            dated_training[
                "completion_date"
            ]
            .ge(
                dated_training[
                    "start_date"
                ]
            )
            .all()
        ),
        "completion does not occur after employment": (
            dated_training[
                "completion_date"
            ]
            .le(
                dated_training[
                    "employment_end_date"
                ]
            )
            .all()
        ),
    },
    name="passed",
)

date_checks

training does not start before hire                   True
training does not start after employment              True
completed and failed records have completion dates    True
incomplete records have no completion dates           True
completion does not occur before start                True
completion does not occur after employment            True
Name: passed, dtype: bool

## 5. Hours and assessment checks

In [7]:
completed_mask = (
    training_details[
        "completion_status"
    ]
    == "Completed"
)

failed_mask = (
    training_details[
        "completion_status"
    ]
    == "Failed"
)

hours_and_score_checks = pd.Series(
    {
        "training hours are nonnegative": (
            training_details[
                "training_hours"
            ].ge(0).all()
        ),
        "completed records meet required hours": (
            training_details.loc[
                completed_mask,
                "training_hours",
            ]
            .ge(
                training_details.loc[
                    completed_mask,
                    "required_hours",
                ]
            )
            .all()
        ),
        "failed records are below required hours": (
            training_details.loc[
                failed_mask,
                "training_hours",
            ]
            .lt(
                training_details.loc[
                    failed_mask,
                    "required_hours",
                ]
            )
            .all()
        ),
        "incomplete records are below required hours": (
            training_details.loc[
                incomplete_mask,
                "training_hours",
            ]
            .lt(
                training_details.loc[
                    incomplete_mask,
                    "required_hours",
                ]
            )
            .all()
        ),
        "completed scores are between 70 and 100": (
            training_details.loc[
                completed_mask,
                "score",
            ]
            .between(
                70,
                100,
            )
            .all()
        ),
        "failed scores are below 70": (
            training_details.loc[
                failed_mask,
                "score",
            ]
            .between(
                0,
                69.9,
            )
            .all()
        ),
        "incomplete records have no score": (
            training_details.loc[
                incomplete_mask,
                "score",
            ]
            .isna()
            .all()
        ),
    },
    name="passed",
)

hours_and_score_checks

training hours are nonnegative                 True
completed records meet required hours          True
failed records are below required hours        True
incomplete records are below required hours    True
completed scores are between 70 and 100        True
failed scores are below 70                     True
incomplete records have no score               True
Name: passed, dtype: bool

## 6. Onboarding coverage

In [8]:
onboarding_records = (
    training_details[
        training_details[
            "program_category"
        ]
        == "Onboarding"
    ]
)

onboarding_counts = (
    onboarding_records
    .groupby("employee_id")
    .size()
    .reindex(
        employees[
            "employee_id"
        ],
        fill_value=0,
    )
)

print(
    "Total onboarding records:",
    len(onboarding_records),
)

print(
    "Minimum onboarding records per employee:",
    onboarding_counts.min(),
)

print(
    "Maximum onboarding records per employee:",
    onboarding_counts.max(),
)

Total onboarding records: 10000
Minimum onboarding records per employee: 1
Maximum onboarding records per employee: 1


In [9]:
onboarding_check = pd.Series(
    {
        "every employee has exactly one onboarding record": (
            onboarding_counts.eq(1).all()
        )
    },
    name="passed",
)

onboarding_check

every employee has exactly one onboarding record    True
Name: passed, dtype: bool

## 7. Leadership-training coverage

In [10]:
manager_levels = {
    "Department Head",
    "Senior Manager",
    "Team Manager",
}

manager_ids = set(
    employees.loc[
        employees[
            "organizational_level"
        ].isin(manager_levels),
        "employee_id",
    ]
)

leadership_employee_ids = set(
    training_details.loc[
        training_details[
            "program_category"
        ]
        == "Leadership",
        "employee_id",
    ]
)

leadership_check = pd.Series(
    {
        "every manager has leadership training": (
            manager_ids
            .issubset(
                leadership_employee_ids
            )
        )
    },
    name="passed",
)

leadership_check

every manager has leadership training    True
Name: passed, dtype: bool

## 8. Safety-training coverage

In [11]:
safety_departments = {
    2,
    3,
}

safety_required_ids = set(
    employees.loc[
        employees[
            "department_id"
        ].isin(
            safety_departments
        )
        | employees[
            "employment_type"
        ].eq("Hourly"),
        "employee_id",
    ]
)

safety_employee_ids = set(
    training_details.loc[
        training_details[
            "program_category"
        ]
        == "Safety",
        "employee_id",
    ]
)

safety_check = pd.Series(
    {
        "safety-required employees have safety training": (
            safety_required_ids
            .issubset(
                safety_employee_ids
            )
        )
    },
    name="passed",
)

safety_check

safety-required employees have safety training    True
Name: passed, dtype: bool

## 9. Training summary

In [12]:
category_summary = (
    training_details
    .groupby(
        "program_category"
    )
    .agg(
        training_records=(
            "training_record_id",
            "count",
        ),
        employees_trained=(
            "employee_id",
            "nunique",
        ),
        average_training_hours=(
            "training_hours",
            "mean",
        ),
        average_score=(
            "score",
            "mean",
        ),
    )
    .round(2)
)

category_summary

,training_records,employees_trained,average_training_hours,average_score
program_category,,,,
Leadership,842,842,28.16,84.51
Onboarding,10000,10000,8.35,85.65
Safety,5421,5421,6.16,85.34
Technical,10170,7395,17.73,84.61


In [13]:
completion_summary = (
    training_details
    .groupby(
        [
            "program_category",
            "completion_status",
        ]
    )
    .size()
    .rename("record_count")
    .reset_index()
)

completion_summary

,program_category,completion_status,record_count
0,Leadership,Completed,751
1,Leadership,Failed,28
2,Leadership,Incomplete,63
3,Onboarding,Completed,9504
4,Onboarding,Failed,101
5,Onboarding,Incomplete,395
6,Safety,Completed,5019
7,Safety,Failed,88
8,Safety,Incomplete,314
9,Technical,Completed,8693


In [14]:
completion_rates = (
    training_details
    .assign(
        completed=(
            training_details[
                "completion_status"
            ]
            == "Completed"
        )
    )
    .groupby(
        "program_category"
    )
    .agg(
        record_count=(
            "training_record_id",
            "count",
        ),
        completion_rate=(
            "completed",
            "mean",
        ),
    )
)

completion_rates[
    "completion_rate"
] = (
    completion_rates[
        "completion_rate"
    ]
    * 100
).round(2)

completion_rates

,record_count,completion_rate
program_category,,
Leadership,842,89.19
Onboarding,10000,95.04
Safety,5421,92.58
Technical,10170,85.48


## 10. Complete validation summary

In [15]:
basic_checks = pd.Series(
    {
        "table has eight columns": (
            len(
                training_records.columns
            )
            == 8
        ),
        "completion statuses are valid": (
            set(
                training_records[
                    "completion_status"
                ]
            )
            .issubset(
                {
                    "Completed",
                    "Incomplete",
                    "Failed",
                }
            )
        ),
    },
    name="passed",
)

all_checks = pd.concat(
    [
        basic_checks,
        key_checks,
        date_checks,
        hours_and_score_checks,
        onboarding_check,
        leadership_check,
        safety_check,
    ]
)

validation_results = pd.DataFrame(
    {
        "check": all_checks.index,
        "passed": all_checks.values,
    }
)

validation_results

,check,passed
0,table has eight columns,True
1,completion statuses are valid,True
2,training-record IDs are complete,True
3,training-record IDs are unique,True
4,employee IDs are valid,True
5,program IDs are valid,True
6,employee-program combinations are unique,True
7,all employees have training records,True
8,training does not start before hire,True
9,training does not start after employment,True


In [16]:
if validation_results["passed"].all():
    print(
        "All training-record validation "
        "checks passed."
    )
else:
    print(
        "One or more training-record "
        "validation checks failed."
    )

All training-record validation checks passed.


## 11. Conclusions

The synthetic training-record table successfully represents employee learning and development activity.

### Successful checks

- Training-record IDs are complete and unique.
- Employee and program foreign keys are valid.
- Every employee has at least one training record.
- Every employee has exactly one onboarding record.
- Managers receive leadership training.
- Safety-required employee groups receive safety training.
- Training dates remain inside employment periods.
- Completed and failed programs have completion dates.
- Incomplete programs do not have completion dates.
- Completed programs meet required-hour requirements.
- Assessment scores agree with completion status.
- All training-record validation checks passed.

### Current simplifications

- Employees can complete each program only once.
- Recurring annual safety training is not yet represented.
- Training assignments are based primarily on department, employment type, organizational level, and tenure.
- Training outcomes are synthetic and do not yet depend on performance-review history.
- Training completion does not yet automatically create promotion or employee-event records.